In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import re

spark = (
    SparkSession.builder
    .appName("nyc_taxi_project")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

data_dir = "data"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/22 20:45:02 WARN Utils: Your hostname, SD-02044.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.19 instead (on interface en0)
26/11/22 20:45:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/d.makarova/projects/nyc_taxi_project/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/22 20:45:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
trips_raw = spark.read.parquet(f"{data_dir}/yellow_tripdata_*.parquet")
zones_raw = spark.read.csv(f"{data_dir}/taxi_zone_lookup.csv", header=True, inferSchema=True)
holidays_raw = spark.read.csv(f"{data_dir}/us_holidays.csv", header=True, inferSchema=True)
weather_raw = spark.read.csv(f"{data_dir}/nyc_weather.csv", header=True, inferSchema=True)

26/11/22 20:45:05 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File data/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.cat

In [3]:
def preview(df):
    df.show(5, truncate=False)
    df.printSchema()

preview(trips_raw)
preview(zones_raw)
preview(holidays_raw)
preview(weather_raw)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-10-01 00:15:32 |2025-10-01 01:04:03  |1              |17.2         |2         |N                 |132         |107 

In [4]:
def to_snake_case(name):
    name = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", "_", name)
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return name.lower()

def rename_to_snake(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

trips = rename_to_snake(trips_raw)
zones = rename_to_snake(zones_raw)
holidays = rename_to_snake(holidays_raw)
weather = rename_to_snake(weather_raw)

print(trips.columns)
print(zones.columns)
print(holidays.columns)
print(weather.columns)

['vendor_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'ratecode_id', 'store_and_fwd_flag', 'pu_location_id', 'do_location_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee']
['location_id', 'borough', 'zone', 'service_zone']
['date', 'holiday_name']
['time', 'temp', 'prcp', 'wspd']


In [5]:
total_rows = trips.count()
print(f"Всего поездок: {total_rows:,}")

Всего поездок: 23,992,355


In [6]:
# Список id зон
ids = []
for row in zones.select("location_id").collect():
    ids.append(row["location_id"])

# Правило 1: время высадки позже времени посадки
step1 = trips.filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
count1 = step1.count()

# Правило 2: длительность не больше 6 часов
step2 = step1.filter(
    F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime") + F.expr("INTERVAL 6 HOURS")
)
count2 = step2.count()

# Правило 3: расстояние больше 0 и не больше 100
step3 = step2.filter((F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100))
count3 = step3.count()

# Правило 4: пассажиров от 0 до 6, пустое значение допустимо
step4 = step3.filter(
    F.col("passenger_count").isNull() | F.col("passenger_count").between(0, 6)
)
count4 = step4.count()

# Правило 5: денежные поля неотрицательные (пустое значение допустимо)
money_cols = [
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "airport_fee", "cbd_congestion_fee",
]

money_condition = F.lit(True)
for c in money_cols:
    money_condition = money_condition & (F.col(c).isNull() | (F.col(c) >= 0))

step5 = step4.filter(money_condition)
count5 = step5.count()

# Правило 6: ID зон есть в справочнике
step6 = step5.filter(
    F.col("pu_location_id").isin(ids) & F.col("do_location_id").isin(ids)
)
count6 = step6.count()

# Правило 7 (дополнительное): дата посадки в периоде данных
step7 = step6.filter(
    F.to_date("tpep_pickup_datetime").between("2025-10-01", "2026-03-31")
)
count7 = step7.count()

trips_clean = step7
clean_rows = count7

In [7]:
removed1 = total_rows - count1
removed2 = count1 - count2
removed3 = count2 - count3
removed4 = count3 - count4
removed5 = count4 - count5
removed6 = count5 - count6
removed7 = count6 - count7

quality_report = spark.createDataFrame(
    [
        ("dropoff_after_pickup", removed1, round(removed1 / total_rows * 100, 4)),
        ("duration_max_6h", removed2, round(removed2 / total_rows * 100, 4)),
        ("distance_0_100", removed3, round(removed3 / total_rows * 100, 4)),
        ("passenger_count_0_6", removed4, round(removed4 / total_rows * 100, 4)),
        ("money_non_negative", removed5, round(removed5 / total_rows * 100, 4)),
        ("zone_ids_in_lookup", removed6, round(removed6 / total_rows * 100, 4)),
        ("pickup_in_period", removed7, round(removed7 / total_rows * 100, 4)),
    ],
    ["filter_name", "rows_removed", "share_of_total"],
)

quality_report.show(truncate=False)

print(f"Было: {total_rows:,}")
print(f"Осталось: {clean_rows:,} ({clean_rows / total_rows * 100:.2f}%)")
print(f"Удалено всего: {total_rows - clean_rows:,}")

+--------------------+------------+--------------+
|filter_name         |rows_removed|share_of_total|
+--------------------+------------+--------------+
|dropoff_after_pickup|322910      |1.3459        |
|duration_max_6h     |7409        |0.0309        |
|distance_0_100      |749951      |3.1258        |
|passenger_count_0_6 |17          |1.0E-4        |
|money_non_negative  |784713      |3.2707        |
|zone_ids_in_lookup  |0           |0.0           |
|pickup_in_period    |15          |1.0E-4        |
+--------------------+------------+--------------+

Было: 23,992,355
Осталось: 22,127,340 (92.23%)
Удалено всего: 1,865,015


In [8]:
trips.select(
    "trip_distance", "passenger_count", "fare_amount", "tip_amount", "total_amount"
).summary("min", "50%", "max").show()

trips.select(
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup"),
).show()

+-------+-------------+---------------+-----------+----------+------------+
|summary|trip_distance|passenger_count|fare_amount|tip_amount|total_amount|
+-------+-------------+---------------+-----------+----------+------------+
|    min|          0.0|              0|    -2555.2|   -333.33|     -2560.2|
|    50%|         1.84|              1|       14.9|       2.0|       22.74|
|    max|     328522.2|              9|     2555.2|     800.0|      2560.2|
+-------+-------------+---------------+-----------+----------+------------+

+-------------------+-------------------+
|         min_pickup|         max_pickup|
+-------------------+-------------------+
|2008-12-31 23:03:20|2026-04-01 00:06:25|
+-------------------+-------------------+



In [9]:
# Справочник зон
zones_pu = zones.select(
    F.col("location_id").alias("pu_location_id"),
    F.col("borough").alias("pu_borough"),
    F.col("zone").alias("pu_zone"),
)
zones_do = zones.select(
    F.col("location_id").alias("do_location_id"),
    F.col("borough").alias("do_borough"),
    F.col("zone").alias("do_zone"),
)

holidays_prep = (
    holidays
    .select(F.col("date").alias("pickup_date"), "holiday_name")
    .dropDuplicates(["pickup_date"])
)
weather_prep = (
    weather
    .select(F.col("time").alias("pickup_date"), "temp", "prcp", "wspd")
    .dropDuplicates(["pickup_date"])
)

In [10]:
trips_enriched = (
    trips_clean
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .join(F.broadcast(zones_pu), on="pu_location_id", how="left")
    .join(F.broadcast(zones_do), on="do_location_id", how="left")
    .join(F.broadcast(holidays_prep), on="pickup_date", how="left")
    .join(F.broadcast(weather_prep), on="pickup_date", how="left")
    .withColumn("is_holiday", F.col("holiday_name").isNotNull())
    .withColumn(
        "day_type",
        F.when(F.col("is_holiday"), "holiday")
         .when(F.dayofweek("pickup_date").isin(1, 7), "weekend")
         .otherwise("weekday"),
    )
     .withColumn(
        "weather_type",
        F.when(F.col("prcp").isNull(), "unknown")
         .when(F.col("prcp") > 0, "precip")
         .otherwise("clear"),
    )
)

In [11]:
trips_enriched.cache()

enriched_rows = trips_enriched.count()
print(f"Строк в trips_clean: {clean_rows:,}")
print(f"Строк в trips_enriched: {enriched_rows:,}")

26/11/22 20:45:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 52:=============================================>            (7 + 2) / 9]

Строк в trips_clean: 22,127,340
Строк в trips_enriched: 22,127,340


In [12]:
trips_enriched.select(
    "pickup_date", "pu_zone", "do_zone", "day_type", "holiday_name", "prcp", "weather_type"
).show(5, truncate=False)

print("Поездок без данных об осадках:", trips_enriched.filter(F.col("prcp").isNull()).count())

trips_enriched.filter(F.col("prcp").isNull()).groupBy("pickup_date").count().orderBy("pickup_date").show()

print("Строк в метеосводке:", weather.count())
weather.filter(F.col("prcp").isNull()).show()

+-----------+-------------------------+-----------------------------+--------+------------+----+------------+
|pickup_date|pu_zone                  |do_zone                      |day_type|holiday_name|prcp|weather_type|
+-----------+-------------------------+-----------------------------+--------+------------+----+------------+
|2025-10-01 |JFK Airport              |Gramercy                     |weekday |NULL        |0.0 |clear       |
|2025-10-01 |Yorkville West           |Sutton Place/Turtle Bay North|weekday |NULL        |0.0 |clear       |
|2025-10-01 |SoHo                     |TriBeCa/Civic Center         |weekday |NULL        |0.0 |clear       |
|2025-10-01 |Times Sq/Theatre District|Manhattan Valley             |weekday |NULL        |0.0 |clear       |
|2025-10-01 |Upper East Side South    |Lincoln Square East          |weekday |NULL        |0.0 |clear       |
+-----------+-------------------------+-----------------------------+--------+------------+----+------------+
only showi

In [13]:
top_routes = (
    trips_enriched
    .groupBy("pu_zone", "do_zone")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    )
    .orderBy(F.desc("trips"))
    .limit(10)
)

top_routes.show(truncate=False)

[Stage 67:===================>                                      (3 + 6) / 9]

+---------------------+---------------------+------+-------------+
|pu_zone              |do_zone              |trips |total_revenue|
+---------------------+---------------------+------+-------------+
|Upper East Side South|Upper East Side North|152516|2473424.75   |
|Upper East Side North|Upper East Side South|131568|2206536.96   |
|Upper East Side South|Upper East Side South|101662|1486553.64   |
|Upper East Side North|Upper East Side North|97463 |1328081.74   |
|Midtown Center       |Upper East Side South|66777 |1289822.31   |
|Upper East Side South|Midtown Center       |62061 |1186989.61   |
|Midtown Center       |Upper East Side North|53629 |1304861.72   |
|Lincoln Square East  |Upper West Side South|52708 |814587.0     |
|Upper West Side South|Upper West Side North|51762 |734915.42    |
|Upper West Side South|Lincoln Square East  |50319 |758323.19    |
+---------------------+---------------------+------+-------------+



In [14]:
place_stat = (
    trips_enriched
    .groupBy("pu_borough")
    .agg(F.count("*").alias("trips"))
    .withColumn("trips_share", F.round(F.col("trips") / enriched_rows * 100, 2))
    .orderBy(F.desc("trips"))
)

place_stat.show(truncate=False)

[Stage 70:======================================>                   (6 + 3) / 9]

+-------------+--------+-----------+
|pu_borough   |trips   |trips_share|
+-------------+--------+-----------+
|Manhattan    |19052336|86.1       |
|Queens       |2003328 |9.05       |
|Brooklyn     |831811  |3.76       |
|Bronx        |203220  |0.92       |
|Unknown      |29061   |0.13       |
|N/A          |4447    |0.02       |
|Staten Island|2300    |0.01       |
|EWR          |837     |0.0        |
+-------------+--------+-----------+



In [15]:
def compare_by(df, group_col):
    daily = (
        df.groupBy("pickup_date", group_col)
        .agg(F.count("*").alias("trips"))
    )

    per_day = (
        daily.groupBy(group_col)
        .agg(
            F.count("*").alias("days_count"),
            F.round(F.avg("trips"), 0).alias("avg_trips_per_day"),
        )
    )

    is_card = (F.col("payment_type") == 1) & (F.col("fare_amount") > 0)

    per_trip = (
        df.groupBy(group_col)
        .agg(
            F.round(F.avg("total_amount"), 2).alias("avg_total"),
            F.round(
                F.avg(F.when(is_card, F.col("tip_amount") / F.col("fare_amount") * 100)),
                2,
            ).alias("avg_tip_percent"),
        )
    )

    return (
        per_day.join(per_trip, on=group_col)
        .select(group_col, "days_count", "avg_trips_per_day", "avg_total", "avg_tip_percent")
        .orderBy(group_col)
    )

In [16]:
weather_comparison = compare_by(
    trips_enriched.filter(F.col("weather_type") != "unknown"),
    "weather_type",
)
weather_comparison.show()

+------------+----------+-----------------+---------+---------------+
|weather_type|days_count|avg_trips_per_day|avg_total|avg_tip_percent|
+------------+----------+-----------------+---------+---------------+
|       clear|        90|         125341.0|    30.08|          24.87|
|      precip|        89|         117429.0|    29.76|          24.84|
+------------+----------+-----------------+---------+---------------+



In [17]:
day_type_comparison = compare_by(trips_enriched, "day_type")
day_type_comparison.show()

[Stage 83:>                                                         (0 + 8) / 9]

+--------+----------+-----------------+---------+---------------+
|day_type|days_count|avg_trips_per_day|avg_total|avg_tip_percent|
+--------+----------+-----------------+---------+---------------+
| holiday|        13|         101153.0|    29.27|          25.22|
| weekday|       118|         123199.0|    30.45|          24.89|
| weekend|        51|         123036.0|    28.92|          24.66|
+--------+----------+-----------------+---------+---------------+



In [18]:
monthly = (
    trips_enriched
    .withColumn("month", F.trunc("pickup_date", "month"))
    .groupBy("pu_zone", "month")
    .agg(F.count("*").alias("trips_curr"))
)

w = Window.partitionBy("pu_zone").orderBy("month")

mom = (
    monthly
    .withColumn("trips_prev", F.lag("trips_curr").over(w))
    .withColumn("prev_month", F.lag("month").over(w))
    .withColumn(
        "mom_change_pct",
        F.round((F.col("trips_curr") - F.col("trips_prev")) / F.col("trips_prev") * 100, 2),
    )
)

top5_growth = (
    mom
    .filter(F.col("trips_prev") >= 100)
    .orderBy(F.desc("mom_change_pct"))
    .limit(5)
    .select("pu_zone", "month", "trips_prev", "trips_curr", "mom_change_pct")
)

top5_growth.show(truncate=False)

[Stage 91:>                                                         (0 + 8) / 9]

+------------------+----------+----------+----------+--------------+
|pu_zone           |month     |trips_prev|trips_curr|mom_change_pct|
+------------------+----------+----------+----------+--------------+
|Dyker Heights     |2025-12-01|373       |879       |135.66        |
|Battery Park      |2026-03-01|489       |827       |69.12         |
|Old Astoria       |2025-12-01|2125      |3590      |68.94         |
|Inwood Hill Park  |2025-12-01|115       |194       |68.7          |
|Brooklyn Navy Yard|2026-03-01|524       |874       |66.79         |
+------------------+----------+----------+----------+--------------+



In [19]:
daily_zone = (
    trips_enriched
    .groupBy("pu_zone", "pickup_date", "holiday_name", "weather_type")
    .agg(F.count("*").alias("trips"))
    .withColumn("day_num", F.datediff("pickup_date", F.to_date(F.lit("2025-01-01"))))
)

In [20]:
w7 = Window.partitionBy("pu_zone").orderBy("day_num").rangeBetween(-6, 0)

zone_days = (
    daily_zone
    .withColumn("rolling_7d_avg", F.avg("trips").over(w7))
    .withColumn("rolling_7d_std", F.stddev_samp("trips").over(w7))
    .filter(F.col("rolling_7d_std") > 0)
    .withColumn(
        "deviation_sigma",
        (F.col("trips") - F.col("rolling_7d_avg")) / F.col("rolling_7d_std"),
    )
)

In [21]:
anomalies = (
    zone_days
    .filter(F.abs("deviation_sigma") > 2)
    .withColumn("is_holiday", F.col("holiday_name").isNotNull())
    .withColumn("is_precip", F.col("weather_type") == "precip")
    .select(
        "pu_zone",
        "pickup_date",
        "trips",
        "rolling_7d_avg",
        "rolling_7d_std",
        "deviation_sigma",
        "is_holiday",
        "holiday_name",
        "is_precip",
    )
)

anomalies.cache()
print("Всего аномальных пар (зона, день):", anomalies.count())

anomalies.orderBy(F.desc(F.abs("deviation_sigma"))).show(20, truncate=False)

[Stage 97:===================>                                      (3 + 6) / 9]

Всего аномальных пар (зона, день): 564
+-------------------------------+-----------+-----+------------------+-------------------+------------------+----------+----------------+---------+
|pu_zone                        |pickup_date|trips|rolling_7d_avg    |rolling_7d_std     |deviation_sigma   |is_holiday|holiday_name    |is_precip|
+-------------------------------+-----------+-----+------------------+-------------------+------------------+----------+----------------+---------+
|Broad Channel                  |2026-02-25 |2    |1.1428571428571428|0.37796447300922725|2.2677868380553634|false     |NULL            |true     |
|Marine Park/Floyd Bennett Field|2026-03-22 |3    |2.142857142857143 |0.37796447300922725|2.2677868380553634|false     |NULL            |true     |
|Astoria Park                   |2025-10-07 |3    |1.2857142857142858|0.7559289460184545 |2.267786838055363 |false     |NULL            |false    |
|Bloomfield/Emerson Hill        |2025-12-31 |3    |1.2857142857142858|0.7

In [22]:
(
    anomalies
    .groupBy("pickup_date", "holiday_name", "is_precip")
    .agg(
        F.count("*").alias("anomalous_zones"),
        F.sum(F.when(F.col("deviation_sigma") > 0, 1).otherwise(0)).alias("spikes"),
        F.sum(F.when(F.col("deviation_sigma") < 0, 1).otherwise(0)).alias("drops"),
    )
    .orderBy(F.desc("anomalous_zones"))
    .show(15, truncate=False)
)

# Какая доля аномалий совпадает с праздниками и осадками
anomalies.groupBy("is_holiday", "is_precip").count().orderBy("is_holiday", "is_precip").show()

+-----------+---------------------+---------+---------------+------+-----+
|pickup_date|holiday_name         |is_precip|anomalous_zones|spikes|drops|
+-----------+---------------------+---------+---------------+------+-----+
|2026-01-10 |NULL                 |true     |38             |38    |0    |
|2026-01-01 |New Year's Day       |true     |27             |27    |0    |
|2026-01-25 |NULL                 |true     |26             |7     |19   |
|2025-12-31 |New Year's Eve       |false    |25             |24    |1    |
|2025-10-31 |Halloween            |true     |17             |17    |0    |
|2026-02-23 |NULL                 |true     |16             |0     |16   |
|2026-02-16 |Washington's Birthday|false    |14             |1     |13   |
|2025-11-16 |NULL                 |true     |13             |13    |0    |
|2026-02-07 |NULL                 |true     |12             |10    |2    |
|2025-12-14 |NULL                 |true     |12             |11    |1    |
|2025-11-27 |Thanksgiving

In [23]:
weather.filter(
    F.col("time").isin("2026-01-10", "2026-01-25", "2026-02-23", "2025-10-20")
).orderBy("time").show()

+----------+----+----+----+
|      time|temp|prcp|wspd|
+----------+----+----+----+
|2025-10-20|15.6|24.7|25.8|
|2026-01-10| 5.2|10.3|11.8|
|2026-01-25|-7.5|29.1|18.8|
|2026-02-23|-1.7|49.9|29.5|
+----------+----+----+----+



In [24]:
OUTPUT_PATH = "output/trips_enriched"

(
    trips_enriched
    .write
    .mode("overwrite")
    .partitionBy("pickup_year", "pickup_month")
    .parquet(OUTPUT_PATH)
)

In [25]:
jan = (
    spark.read.parquet(OUTPUT_PATH)
    .filter((F.col("pickup_year") == 2026) & (F.col("pickup_month") == 1))
)

top_routes_jan = (
    jan
    .groupBy("pu_zone", "do_zone")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    )
    .orderBy(F.desc("trips"))
    .limit(10)
)

top_routes_jan.explain(mode="formatted")
top_routes_jan.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (7)
+- TakeOrderedAndProject (6)
   +- HashAggregate (5)
      +- Exchange (4)
         +- HashAggregate (3)
            +- Project (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [total_amount#14113, pu_zone#14118, do_zone#14120, pickup_year#14128, pickup_month#14129]
Batched: true
Location: InMemoryFileIndex [file:/Users/d.makarova/projects/nyc_taxi_project/output/trips_enriched]
PartitionFilters: [isnotnull(pickup_year#14128), isnotnull(pickup_month#14129), (pickup_year#14128 = 2026), (pickup_month#14129 = 1)]
ReadSchema: struct<total_amount:double,pu_zone:string,do_zone:string>

(2) Project
Output [3]: [total_amount#14113, pu_zone#14118, do_zone#14120]
Input [5]: [total_amount#14113, pu_zone#14118, do_zone#14120, pickup_year#14128, pickup_month#14129]

(3) HashAggregate
Input [3]: [total_amount#14113, pu_zone#14118, do_zone#14120]
Keys [2]: [pu_zone#14118, do_zone#14120]
Functions [2]: [partial_count(1), partial_sum(tota